# 03 — Excedencias normativas (Res. 2254/2017 y OMS)

Demo en navegador de `exceedance_report`: ¿cuántas veces y por cuánto se superan los límites
colombianos y de la OMS para PM2.5?

In [ ]:
import piplite
await piplite.install(["estadistica-ambiental", "pymannkendall"])

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 24 * 30  # 30 días horarios
fechas = pd.date_range("2024-06-01", periods=n, freq="h")
ciclo = 18 + 8 * np.sin(2 * np.pi * np.arange(n) / 24)
# Picos sintéticos para simular un episodio de quema agrícola.
ciclo[200:260] += 40
ciclo[1100:1140] += 30
pm25 = np.clip(ciclo + rng.normal(0, 3, size=n), 0, None)
serie = pd.Series(pm25, index=fechas, name="pm25")
serie.describe().round(2)

## `exceedance_report` — tabla normativa CO + OMS

In [ ]:
from estadistica_ambiental.inference.intervals import exceedance_report

reporte = exceedance_report(serie, variable="pm25")
reporte

## Visualización del umbral diario

Resampleamos a 24 h y graficamos contra el límite diario de la Res. 2254/2017 (37 µg/m³).

In [ ]:
import matplotlib.pyplot as plt

diario = serie.resample("D").mean()
fig, ax = plt.subplots(figsize=(9, 3.5))
diario.plot(ax=ax, label="PM2.5 promedio diario")
ax.axhline(37, color="orange", linestyle="--", label="Res. 2254/2017 (37 µg/m³)")
ax.axhline(15, color="red", linestyle=":", label="OMS 2021 (15 µg/m³)")
ax.set_ylabel("µg/m³")
ax.legend()
plt.show()